## Архитектура

### Какую проблему решает инструмент
Ручной аудит кода занимает много времени и часто дает несистемный результат: один файл проверили глубоко, другой поверхностно, а связи между ними потерялись. Этот агент решает именно эту задачу: делает анализ последовательным, воспроизводимым и документируемым.

### Что делает агент в проекте
1. Собирает поддерживаемые файлы (`.py`, `.cpp`, `.java`, `.sql`).
2. Запрашивает у модели стартовый план анализа.
3. Анализирует файлы по очереди, а при признаках риска запускает углубленный проход (`deep_dive`).
4. Формирует сводку по файлам, общий вывод и рефлексию.
5. Сохраняет итог в Markdown-отчет, который можно сразу отдавать команде.

### Какие технологии используются и зачем
1. `agent.py` — оркестрация процесса, чтобы не писать отдельные скрипты под каждый запуск.
2. `analysis_api_base.py` — общая логика анализа: language detection, чанкинг, промпты, память. Это единая «платформа», поверх которой подключаются разные модели.
3. `memory.py` + `chroma_db/` — хранение контекста между файлами, чтобы ответы были связными, а не изолированными.
4. `prompts/` — промпты по языкам и этапам (plan/reflect), чтобы поведение было управляемым и прозрачным.
5. `reporter.py` — генерация читабельного Markdown-результата для разработчиков и тимлидов.

Идея демонстрации простая: мы не придумываем новый код в ноутбуке, а запускаем действующий pipeline проекта в том виде, в котором он реально используется.

In [3]:
from pathlib import Path
import os

from agent import Agent

PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/vladislav/Рабочий стол/working/agents/ai_code_analyzer


## Провайдер модели: что выбрать и почему

В проекте уже есть несколько провайдеров, и это важная инженерная часть: логика анализа одна, а модель можно менять под окружение.

1. `gigachat` (`gigachat_api.py`)
Причина выбора: рабочий корпоративный сценарий, когда модель доступна как сервис. Плюс — не нужно держать тяжелые веса локально.

2. `local` (`local_model_api.py`)
Причина выбора: данные остаются в локальном контуре, удобно для закрытых сред и оффлайн-работы. Требует `torch/transformers` и путь к модели.

3. `kobold` (`koboldcpp_api.py`)
Причина выбора: OpenAI-compatible endpoint для локальных/гибридных схем, когда нужно быстро подключить уже поднятый inference-сервер.


In [5]:
SOURCE_PATH = os.getenv('ANALYZE_SOURCE', 'sandbox')
OUTPUT_FILE = os.getenv('ANALYZE_OUTPUT', 'analysis_report_demo.md')
PROVIDER = os.getenv('DEMO_PROVIDER', 'kobold').strip().lower()
MODEL_NAME = os.getenv('DEMO_MODEL')

if PROVIDER == 'gigachat':
    from gigachat_api import GigaChatAPI
    model = GigaChatAPI(model_name=MODEL_NAME or 'GigaChat')
elif PROVIDER == 'local':
    from local_model_api import LocalModelAPI
    model = LocalModelAPI(model_path=MODEL_NAME or os.getenv('LOCAL_MODEL_PATH'))
elif PROVIDER == 'kobold':
    from koboldcpp_api import KoboldCppAPI
    model = KoboldCppAPI(model_name=MODEL_NAME or 'phi')
else:
    raise ValueError(f'Неизвестный провайдер: {PROVIDER}')

model.set_use_sandbox(False)
agent = Agent(model=model)

print('PROVIDER =', PROVIDER)
print('SOURCE_PATH =', SOURCE_PATH)
print('OUTPUT_FILE =', OUTPUT_FILE)

PROVIDER = kobold
SOURCE_PATH = sandbox
OUTPUT_FILE = analysis_report_demo.md


## Полный запуск: что происходит под капотом

После запуска этой ячейки `Agent` выполнит весь цикл автоматически.

1. `ProjectLoader` берет локальный путь проекта.
2. Агент собирает список файлов для анализа.
3. Модель строит исходный план и при необходимости обновляет его после каждого файла.
4. Для каждого файла выполняется блочный анализ, а затем краткий вывод по файлу.
5. Если в ответе есть триггеры риска (`ошибка`, `уязвимость`, `инъекция` и т.д.), агент делает `deep_dive`.
6. В конце формируется общий вывод по проекту и рефлексия.
7. `reporter.py` сохраняет итоговый отчет в Markdown.

Это дает практическую ценность: на выходе не просто «ответ модели», а структурированный артефакт с понятной трассировкой по файлам.

In [6]:
source = Path(SOURCE_PATH)
if not source.exists():
    raise FileNotFoundError(f'Путь для анализа не найден: {source.resolve()}')

agent.run_from_path(str(source), output_file=OUTPUT_FILE)
print('Готово. Отчет сохранен в', OUTPUT_FILE)

[loader] Используем локальный проект: /home/vladislav/Рабочий стол/working/agents/ai_code_analyzer/sandbox
[agent] Файлов для анализа: 4
[agent] План анализа:
1. Проверка schema.sql:
   a. Убедитесь, что структура базы данных содержит необходимые таблицы с правильными столпами, индексами и ограничениями.
   b. Ищите логические ошибки в определениях таблиц, таких как несоответствующие типы данных или неправильные связи между таблицами.
   c. Оцените, есть ли уникальные ограничения (PRIMARY KEY, UNIQUE) для обеспечения данных уникальности.
   d. Проверьте целостность определений таблиц, чтобы убедиться, что они корректно представляют реляционную модель.

2. Проверка app.py:
   a. Проверьте процесс загрузки и выполнения schema.sql, убедитесь, что база данных создается корректно.
   b. Изучите код CRUD-operationen (Create, Read, Update, Delete) и убедитесь, что они правильно обрабатывают данные в соответствии с базой данных.
   c. Оцените обработку ошибок и исключения, чтобы убедиться, что

KeyboardInterrupt: 

In [7]:
report_path = Path(OUTPUT_FILE)
report_text = report_path.read_text(encoding='utf-8')

print(report_text[:4000])
if len(report_text) > 4000:
    print('\n... (показан фрагмент отчета)')

# Отчет анализа кода

## Обнаруженные проблемы по файлам:

### Файл: `schema.sql`

#### Блок 1
Существенных проблем не обнаружено в текущем блоке.

#### Краткий вывод по файлу
Критичных проблем по файлу не обнаружено.

### Файл: `app.py`

#### Блок 1
Найдены проблемы:
- Возможная уязвимость: используется eval(), это риск выполнения произвольного кода.
- Логическая ошибка: цикл идет до len(...)+1, возможен выход за границы индекса.

#### Краткий вывод по файлу
Найдены ошибки и уязвимости. Приоритет: убрать eval/хардкод и параметризовать SQL.

### Файл: `auth.py`

#### Блок 1
Найдены проблемы:
- Ошибка обработки исключений: except Exception: pass скрывает реальные проблемы.
- Уязвимость: хардкод пароля в коде аутентификации.

#### Краткий вывод по файлу
Найдены ошибки и уязвимости. Приоритет: убрать eval/хардкод и параметризовать SQL.

### Файл: `storage.py`

#### Блок 1
Найдены проблемы:
- Потенциальная SQL-инъекция: запрос собирается через форматирование строки.

#### Краткий вывод по 

## Целевая архитектура развития: большая + малые модели

Следующий этап развития системы: перейти от одиночного анализа к **распределенному контуру задач**, где у каждой модели своя роль.

### Роли в системе
1. **Большая модель (GigaChat) — Planner/Coordinator**
   Анализирует структуру проекта, формирует общий план и раскладывает его в пул атомарных задач.
2. **Малые локальные модели — Workers**
   Берут короткие специализированные задачи из пула: проверка конкретной функции, блока SQL, граничных условий, поиска паттернов уязвимостей.
3. **Оркестратор (Agent) — Dispatcher/Aggregator**
   Раздает задачи воркерам, собирает ответы, запускает повторную проверку спорных мест и формирует единый отчет.

### Как выглядит поток выполнения
1. `GigaChat` строит глобальный план и создает пул задач (task queue).
2. Локальные модели параллельно забирают задачи небольшого размера.
3. Каждая локальная модель возвращает структурированный результат: `finding`, `severity`, `evidence`, `fix_hint`.
4. Оркестратор агрегирует результаты и отправляет неоднозначные кейсы обратно в `GigaChat` на финальное решение.
5. На выходе формируется итоговый Markdown-отчет с приоритизацией и рекомендациями.

### Зачем такая схема
1. **Скорость**: мелкие задачи выполняются параллельно на легких моделях.
2. **Стоимость**: дорогая большая модель используется только для планирования и арбитража.
3. **Масштабируемость**: можно добавлять воркеры под конкретные типы задач без изменения ядра.
4. **Надежность**: конфликтующие ответы воркеров перепроверяются координатором.

### Что добавить в текущий проект для этого этапа
1. Модуль `task_queue` (очередь задач + статусы `new/in_progress/done/failed`).
2. Формат контракта задачи и результата (`JSON schema` для единообразия).
3. Worker-раннер для локальных моделей с ограничением времени и ретраями.
4. Aggregator-слой для дедупликации находок и объединения evidence.
5. Финальный арбитр на `GigaChat` для сложных/спорных случаев.


<div style="text-align:center; margin-top: 12px;">
  <img src="assets/thanks_attention_plain.svg" alt="Спасибо за внимание" width="900"/>
</div>